# Lesson 4: Deployment

**Complete notebook.** Every cell runs as it is. Run the cells in order, top to
bottom, and read the explanation between them: what matters is what the
output shows and why.

## What this notebook builds

The trained model from Lesson 3, wrapped so that somebody who is not in this room
can use it. A page with an upload button, a public link, and a reply that is
structured data rather than a picture.

Everything runs in Colab. There is no container and no cloud project. What is
being taught is the step from a notebook that works for the person who wrote it
to a thing that survives a stranger, and that step is the same either way: load
the model once, refuse bad input clearly, answer in a shape a caller can parse.

## The order of work

1. Setup and the weights from Lesson 3
2. Load the model once
3. The three functions that are the whole service
4. The interface, and a public link
5. Testing the error conditions
6. Cold and warm response time
7. Cloud or edge

## Timing

The build is short. Most of the lab goes on Section 5, and Section 5 is what the
sheet assesses, so do not let it be the part that gets rushed.

---

## 1. Setup

Gradio builds the page. Ultralytics is the same pinned version as Lesson 3, since
weights and library have to agree.

In [ ]:
ULTRALYTICS_VERSION = "8.4.120"

try:
    import google.colab  # noqa: F401
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

if RUNNING_ON_COLAB:
    import subprocess
    # gradio is left unpinned, because the page only needs a recent version. The
    # version this prints is the one the page was built with.
    subprocess.run(
        ["pip", "install", "-q", "ultralytics==" + ULTRALYTICS_VERSION, "gradio"],
        check=True,
    )

print("Running on Colab:", RUNNING_ON_COLAB)

In [ ]:
import io
import os
import time

import gradio as gr
from PIL import Image, UnidentifiedImageError
from ultralytics import YOLO

print("gradio:", gr.__version__)

from pathlib import Path

repo_candidates = [Path.cwd(), *Path.cwd().parents]
if RUNNING_ON_COLAB:
    repo_candidates.append(Path("/content/c389-ca1-people-counting"))
repo_root = next(
    (root for root in repo_candidates if (root / "models/improved/best.pt").is_file()),
    None,
)

if repo_root is not None:
    PROJECT_ROOT = str(repo_root)
    WEIGHTS = str(repo_root / "models" / "improved" / "best.pt")
else:
    if RUNNING_ON_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_ROOT = "/content/drive/MyDrive/c389 ca1 final (improved model)"
    else:
        PROJECT_ROOT = os.getcwd()
    WEIGHTS = os.path.join(PROJECT_ROOT, "artifacts", "L3", "best.pt")

print("weights:", WEIGHTS)


### If the weights are missing

`best.pt` is written by the Lesson 3 notebook into `artifacts/L3/`. If it is not
there, the cell below falls back to the stock pretrained model so that the rest of
the lab can still be built and tested.

The fallback model detects people and cannot report direction, so every count
comes back under a single `person` class. That is a working service answering
honestly about a model that was never trained for this job, which is a more useful
thing to look at than an error.

In [ ]:
if not os.path.isfile(WEIGHTS):
    raise FileNotFoundError(f"Set WEIGHTS to your trained best.pt. Current path: {WEIGHTS}")
MODEL_PATH = WEIGHTS
USING_TRAINED_MODEL = True


---

## 2. Load the model once

Once, here, at startup, and never inside the function that answers a request.

Loading a YOLO model takes seconds. Inside the handler those seconds are added to
every single call, and a second caller arriving while the first is still waiting
loads a second copy into memory. This is the single most common way a working
model becomes an unusable service, and it is one line in the wrong place.

The class list is read from the weights rather than typed here, so a CA1 twist
that appends a class needs no edit in this notebook.

In [ ]:
MODEL = YOLO(MODEL_PATH)
CLASS_NAMES = [MODEL.names[key] for key in sorted(MODEL.names)]

CONFIDENCE_THRESHOLD = 0.40
MAX_IMAGE_PIXELS = 40_000_000
SERVICE_VERSION = "1.0.0"

print("model loaded from  :", MODEL_PATH)
print("classes            :", CLASS_NAMES)
print("confidence threshold:", CONFIDENCE_THRESHOLD)

---

## 3. The three functions that are the whole service

Validate the input, count what is in it, build the reply. Everything else on this
page is Gradio, and Gradio is replaceable. These three functions are what would
move unchanged into a container on a cloud host or onto a device at the edge.

Keeping them separate from the interface is the reason that claim is true rather
than a thing said in a slide.

In [ ]:
def validate_image(uploaded):
    """Check that what arrived can be turned into an image to run on.

    Args:
        uploaded: whatever the caller sent. A path, a file like object, a PIL
            image, or None when nothing was sent at all

    Returns:
        (image, error), exactly one of which is None. The error is a dict
        carrying a status code and a message the caller could act on

    Requirement:
        Four cases are refused before any inference runs: nothing sent, an empty
        file, a file that is not a readable image, and an image so large it would
        exhaust memory.

        A service that hands a text file to the model and returns a stack trace
        has told the caller nothing. Every refusal here names what was wrong and
        what to send instead.
    """
    if uploaded is None:
        return None, {"status": 400, "error": "No image was sent.",
                      "detail": "Attach a JPEG or PNG and send it again."}

    try:
        if isinstance(uploaded, Image.Image):
            image = uploaded
        elif isinstance(uploaded, (bytes, bytearray)):
            if not uploaded:
                return None, {"status": 400, "error": "The upload was empty.",
                              "detail": "The file carried no data."}
            image = Image.open(io.BytesIO(uploaded))
            image.load()
        else:
            if os.path.getsize(uploaded) == 0:
                return None, {"status": 400, "error": "The upload was empty.",
                              "detail": "The file '%s' is zero bytes."
                                        % os.path.basename(str(uploaded))}
            image = Image.open(uploaded)
            image.load()
    except (UnidentifiedImageError, OSError):
        return None, {"status": 415, "error": "That file is not a readable image.",
                      "detail": "Send a JPEG or PNG."}

    if image.width * image.height > MAX_IMAGE_PIXELS:
        return None, {"status": 413, "error": "The image is too large to process.",
                      "detail": "%d by %d pixels is beyond the limit of %d pixels."
                                % (image.width, image.height, MAX_IMAGE_PIXELS)}

    return image.convert("RGB"), None

In [ ]:
def count_people(image, threshold):
    """Run the model on one image and total the detections by class.

    Args:
        image: a PIL image, already validated
        threshold: detections below this confidence are dropped

    Returns:
        (counts, kept), where counts maps every class name to a whole number

    Requirement:
        The same rule as Lesson 3. Every class appears in counts, including the
        ones at zero, and an image with nobody in it is a valid result of zero
        rather than an error.
    """
    result = MODEL.predict(image, conf=0.01, verbose=False)[0]

    kept = []
    boxes = result.boxes
    if boxes is not None:
        for index in range(len(boxes)):
            confidence = round(float(boxes.conf[index].item()), 3)
            if confidence < threshold:
                continue
            class_id = int(boxes.cls[index].item())
            kept.append({
                "class_name": CLASS_NAMES[class_id],
                "confidence": confidence,
                "box": [round(float(v), 1) for v in boxes.xyxy[index].tolist()],
            })

    counts = {}
    for name in CLASS_NAMES:
        counts[name] = 0
    for detection in kept:
        counts[detection["class_name"]] = counts[detection["class_name"]] + 1

    return counts, kept

In [ ]:
def build_reply(uploaded, threshold=CONFIDENCE_THRESHOLD):
    """The whole service, in one function. Validate, count, answer.

    Returns a dict either way. A caller never has to tell success from failure by
    reading prose: a reply carrying "error" failed, and one carrying "counts" did
    not.

    A CA1 twist that adds a class or an extra field changes this function and
    nothing else, which is why the reply is built in one place.
    """
    image, error = validate_image(uploaded)
    if error is not None:
        return error, None

    started = time.time()
    counts, detections = count_people(image, threshold)
    elapsed = time.time() - started

    reply = {
        "status": 200,
        "counts": counts,
        "confidence_threshold": threshold,
        "model_version": SERVICE_VERSION,
        "trained_model": USING_TRAINED_MODEL,
        "image_size": [image.width, image.height],
        "inference_seconds": round(elapsed, 3),
        "detections": detections,
    }
    # Printed rather than kept, so the Colab output below the cell is this
    # service's log. Section 6 reads it.
    print("counted %s in %.3fs on a %dx%d image"
          % (counts, elapsed, image.width, image.height))
    return reply, image

---

## 4. The interface

Gradio turns a function into a page. It needs to be told what the function takes,
what it gives back, and what to call it.

`share=True` is what produces the public link. Without it the page is reachable
only from inside this runtime, which means it has not been deployed to anybody.

In [ ]:
def annotate(image, detections):
    """Draw the kept boxes, so the reply can be checked by eye as well as read."""
    from PIL import ImageDraw

    marked = image.copy()
    draw = ImageDraw.Draw(marked)
    for detection in detections:
        left, top, right, bottom = detection["box"]
        colour = (214, 69, 65) if detection["class_name"].endswith("left") else (46, 112, 196)
        draw.rectangle([left, top, right, bottom], outline=colour, width=3)
        draw.text((left + 3, max(0, top - 12)),
                  "%s %.2f" % (detection["class_name"], detection["confidence"]),
                  fill=colour)
    return marked


def predict(uploaded, threshold):
    """What the page calls. Returns the annotated image and the reply."""
    reply, image = build_reply(uploaded, threshold)
    if image is None:
        return None, reply
    return annotate(image, reply["detections"]), reply

In [ ]:
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type="pil", label="Walkway photograph"),
        gr.Slider(0.05, 0.95, value=CONFIDENCE_THRESHOLD, step=0.05,
                  label="Confidence threshold"),
    ],
    outputs=[
        gr.Image(label="Detections"),
        gr.JSON(label="Reply"),
    ],
    title="People Counting Assistant",
    description="Upload one photograph. The reply says how many people are "
                "moving left and how many are moving right.",
)

demo.launch(share=True)

Two things worth knowing about that link before it goes in a report.

- It lives for 72 hours and dies the moment this runtime stops. It is a demo
  link, not a deployment.
- Anyone holding it can send photographs to it. There is no account and no key.

Send the link to another team and have them send a photograph taken on a
different phone at a different time of day. A service that has only ever been
called by the person who built it has not been tested.

---

## 5. Testing the error conditions

Five things a caller will do, four of which are not a valid photograph. What is
being checked is not whether the service survives, but whether what comes back
tells the caller what to do next.

These call `build_reply` directly rather than going through the page. That is on
purpose. Gradio's image widget refuses some of these before the function ever
sees them, which is worth noticing in itself: a widget defending a function is
not the same as a function that defends itself, and the function is the part that
would move into a container.

Anything printed above the table is the service logging the calls it served, plus
whatever a library decided to say the first time it was asked for something. The
table itself is printed in one piece underneath, which is the part to read.

In [ ]:
import tempfile

scratch = tempfile.mkdtemp()

text_file = os.path.join(scratch, "not_an_image.txt")
with open(text_file, "w", encoding="utf-8") as handle:
    handle.write("this is not a photograph")

empty_file = os.path.join(scratch, "empty.jpg")
open(empty_file, "wb").close()

huge = Image.new("RGB", (9000, 7000), (200, 200, 200))
huge_file = os.path.join(scratch, "huge.jpg")
huge.save(huge_file, quality=30)

nobody = Image.new("RGB", (640, 480), (214, 210, 202))
nobody_file = os.path.join(scratch, "nobody.jpg")
nobody.save(nobody_file)

CASES = [
    ("nothing sent", None),
    ("a text file", text_file),
    ("an empty file", empty_file),
    ("a very large image", huge_file),
    ("an image with nobody in it", nobody_file),
]

# Collected first, printed afterwards. The service logs a line for every call it
# serves, and a library may print a one-time message the first time it is asked
# for something, so printing the table as it goes puts that chatter through the
# middle of the thing being read.
rows = []
for label, payload in CASES:
    reply, _ = build_reply(payload)
    summary = reply.get("error") or ("counts " + str(reply.get("counts")))
    rows.append((label, reply["status"], summary))

print("")
print("%-28s %-7s %s" % ("case", "status", "what came back"))
for label, status, summary in rows:
    print("%-28s %-7s %s" % (label, status, summary))

Copy the status and the message for all five into Section 2 of the sheet, as
they came back rather than as they were expected to come back. The sheet's last
two rows, an image that is not relevant and people outside the field of view,
are sent through the page instead.

The last row is the one to think hardest about. An image with nobody in it is not
a failure and must not be reported as one. Zero people is a correct answer, and a
caller that receives an error for it cannot tell an empty walkway from a broken
service.

---

## 6. Cold and warm response time

A cold start is what somebody waits through when they are the first to arrive
after the service has been sitting idle, or has just been deployed.

It cannot be measured from in here. A cold start is a whole process starting:
importing the libraries, setting up the GPU, reading the weights off a disk that
has never read them before, and only then answering. This notebook paid every one
of those in Section 2 and will not pay them again, so timing another load in this
process measures a warm service and calls it cold.

The warm number is measured here, where the conditions are right. The cold one is
measured by starting a new Python process that does nothing but import, load and
answer once, and timing the whole thing from outside, which is what a host does
when it starts a service after an idle period.

In [ ]:
# Warm, measured here, where this process is exactly the warm service.
sample = Image.new("RGB", (640, 480), (214, 210, 202))

warm = []
for attempt in range(5):
    started = time.time()
    count_people(sample, CONFIDENCE_THRESHOLD)
    warm.append(time.time() - started)

warm_average = sum(warm) / len(warm)
print("warm inference, average : %.3f s" % warm_average)

In [ ]:
import subprocess
import sys

# A separate process, so nothing this notebook has already done counts. It
# imports, loads the weights and answers one request, and reports where its own
# time went. The figure that matters is the total measured from out here, because
# that is what the caller waits through.
weights = os.path.abspath(MODEL_PATH) if os.path.exists(MODEL_PATH) else MODEL_PATH
probe_path = "/content/cold_start_probe.py"

with open(probe_path, "w", encoding="utf-8") as handle:
    print("import sys, time", file=handle)
    print("launch = float(sys.argv[1])", file=handle)
    print("start = time.time()", file=handle)
    print("from ultralytics import YOLO", file=handle)
    print("from PIL import Image", file=handle)
    print("imported = time.time()", file=handle)
    print("model = YOLO(%r)" % weights, file=handle)
    print("loaded = time.time()", file=handle)
    print("model.predict(Image.new('RGB', (640, 480)), conf=0.01, verbose=False)",
          file=handle)
    print("answered = time.time()", file=handle)
    print("print('  importing the libraries : %.3f s' % (imported - start))", file=handle)
    print("print('  reading the weights     : %.3f s' % (loaded - imported))", file=handle)
    print("print('  answering the first time: %.3f s' % (answered - loaded))", file=handle)
    print("print('ANSWERED %.3f' % (answered - launch))", file=handle)

started = time.time()
probe = subprocess.run([sys.executable, probe_path, repr(started)],
                       capture_output=True, text=True)
process_lifetime = time.time() - started

# The caller waits from the moment the process is launched until it answers. What
# happens after that, releasing the GPU and shutting down, can take seconds and
# nobody is waiting for it, so timing the whole subprocess overstates the wait.
cold_total = process_lifetime
report = []
for line in probe.stdout.splitlines():
    if not line.strip():
        continue
    if line.startswith("ANSWERED "):
        cold_total = float(line.split()[1])
    else:
        report.append(line)

if probe.returncode != 0:
    print("The probe did not run. Its last words:")
    print(probe.stderr.strip()[-600:])
else:
    print("inside a process that has just started:")
    for line in report:
        print(line)
    print("")
    print("somebody arriving at a cold service waits : %.3f s" % cold_total)
    print("somebody arriving at a warm one waits     : %.3f s" % warm_average)
    print("cold is %.0f times the warm figure" % (cold_total / warm_average))
    print("")
    print("the probe process lived for %.3f s in total. The difference is it"
          % process_lifetime)
    print("shutting down and releasing the GPU, which no caller waits for.")

Read where the time went, not only the total. The answering is milliseconds and
everything before it is not, which is the whole shape of the problem: this
service is slow to start and fast to answer.

None of that is a bug, and it is worth knowing which, because a team that does
not know it will spend lab time debugging a service that works.

It also decides something real about where this gets deployed. A host that shuts
an idle service down to save money pays that load cost again on the very next
call, so the first person to use it after a quiet period waits through all of it
while everybody after them waits milliseconds. A host that keeps it warm charges
for the idle time instead. Neither is free, and which one is right depends on how
often the walkway is actually looked at.

The scrollback above this cell is the log: every call printed a line, and reading
that log is how a failure gets diagnosed once the caller is somebody else.

---

## 7. Cloud or edge

The same three functions could run on a small device at the walkway instead of on
a machine somewhere else. The trade is not about which is more modern.

- **Sending a photograph away** costs the round trip and needs a connection that
  works. Every image of a public walkway leaves the site.
- **Running on the device** keeps the images at the site and answers with no
  round trip, and gives up the GPU, the ability to update all five services by
  redeploying, and the memory to run a larger model variant.

Neither is correct in general. What makes the answer defensible is naming the
constraint that decides it for this walkway: the connection, the cost, the number
of frames per second actually needed, or who is allowed to see the photographs.

Nothing in this notebook is specific to Colab or to Gradio. That is what makes
the question real rather than rhetorical.

### Before leaving

- The public link was sent to another team, and a photograph from another phone
  came back with a sensible answer
- All seven rows of the error table are recorded, five from Section 5 and two
  sent through the page, with the status and the message that actually came back
- Cold and warm timings are written down as measured, not as remembered
- The cloud against edge answer names the constraint that decides it, not the
  general advantages of each
- `demo.close()` is run below when the lab finishes, or the runtime keeps serving

In [ ]:
demo.close()
print("The share link is closed.")